In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import itertools
import copy
import random
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# ==========================================
# 0. DETERMINISTIC SEEDING (FOR REPORT REPRODUCIBILITY)
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# ==========================================
# 1. DATA LOADING & PREPROCESSING
# ==========================================
print("Loading and preprocessing data...")
SPEED_PATH = './data/speed_matrix_2015_5mph'

speed_df = pd.read_pickle(SPEED_PATH)
speed_matrix_np = speed_df.to_numpy()

# Global Min-Max Scaling [0, 1]
data_min = np.min(speed_matrix_np)
data_max = np.max(speed_matrix_np)
speed_matrix_scaled = (speed_matrix_np - data_min) / (data_max - data_min)

def create_sliding_windows(data, window_size=12, horizon=1):
    x_offsets = np.arange(window_size)
    y_offsets = np.arange(window_size, window_size + horizon)
    num_samples = data.shape[0] - window_size - horizon + 1

    X, Y = [], []
    for t in range(num_samples):
        X.append(data[t + x_offsets, :])
        Y.append(data[t + y_offsets, :])
    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(Y), dtype=torch.float32)

print("Generating sliding windows...")
X_full, Y_full = create_sliding_windows(speed_matrix_scaled)

def apply_missing_data_mask(X, missing_rate):
    if missing_rate == 0.0: return X
    mask = (torch.rand(X.shape) > missing_rate).float()
    return X * mask

rates = [0.0, 0.1, 0.2, 0.4, 0.8]
scenarios = {f"missing_{int(r*100)}": apply_missing_data_mask(X_full, r) for r in rates}

In [ ]:
# ==========================================
# 2. THE 5 CORE SEQUENCE BLOCKS
# ==========================================
class BaseRNN(nn.Module):
    def __init__(self, in_dim, hid_dim):
        super().__init__()
        self.rnn = nn.RNN(in_dim, hid_dim, batch_first=True)
    def forward(self, x): return self.rnn(x)[0]

class BaseLSTM(nn.Module):
    def __init__(self, in_dim, hid_dim):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hid_dim, batch_first=True)
    def forward(self, x): return self.lstm(x)[0]

class BaseBiLSTM(nn.Module):
    def __init__(self, in_dim, hid_dim):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hid_dim // 2, bidirectional=True, batch_first=True)
    def forward(self, x): return self.lstm(x)[0]

class BaseAttnLSTM(nn.Module):
    def __init__(self, in_dim, hid_dim):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hid_dim, batch_first=True)
        self.attn = nn.Linear(hid_dim, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        attn_weights = F.softmax(self.attn(out), dim=1)
        return out * attn_weights

class BaseConvLSTM(nn.Module):
    def __init__(self, in_dim, hid_dim):
        super().__init__()
        self.conv = nn.Conv1d(in_channels=in_dim, out_channels=hid_dim, kernel_size=3, padding=1)
        self.lstm = nn.LSTM(hid_dim, hid_dim, batch_first=True)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = F.relu(self.conv(x))
        x = x.permute(0, 2, 1)
        return self.lstm(x)[0]

BLOCKS = {'RNN': BaseRNN, 'LSTM': BaseLSTM, 'BiLSTM': BaseBiLSTM, 'AttnLSTM': BaseAttnLSTM, 'ConvLSTM': BaseConvLSTM}





In [ ]:
# ==========================================
# 3. GAN GENERATOR & DISCRIMINATOR
# ==========================================
class TrafficGenerator(nn.Module):
    def __init__(self, block_type, nodes=323, hid_dim=64):
        super().__init__()
        self.encoder = BLOCKS[block_type](nodes, hid_dim)
        self.decoder = nn.Linear(hid_dim, nodes)

    def forward(self, x):
        features = self.encoder(x)
        return self.decoder(features[:, -1, :]).unsqueeze(1)

class TrafficDiscriminator(nn.Module):
    def __init__(self, block_type, nodes=323, hid_dim=32):
        super().__init__()
        self.encoder = BLOCKS[block_type](nodes, hid_dim)
        self.classifier = nn.Linear(hid_dim, 1)

    def forward(self, sequence):
        features = self.encoder(sequence)
        return torch.sigmoid(self.classifier(features[:, -1, :]))

In [ ]:
# ==========================================
# 4. LOSS FUNCTIONS & METRICS
# ==========================================
def masked_mse_loss(preds, targets, null_val=0.0):
    mask = (targets != null_val).float()
    loss = (preds - targets) ** 2
    return torch.mean(loss * mask) / (torch.mean(mask) + 1e-5)

def calculate_metrics(preds, targets, null_val=0.0):
    mask = (targets != null_val).float()
    mask /= torch.mean(mask)
    mask = torch.where(torch.isnan(mask), torch.zeros_like(mask), mask)

    mae_loss = torch.abs(preds - targets) * mask
    mae = torch.mean(torch.where(torch.isnan(mae_loss), torch.zeros_like(mae_loss), mae_loss))

    rmse_loss = ((preds - targets) ** 2) * mask
    rmse = torch.sqrt(torch.mean(torch.where(torch.isnan(rmse_loss), torch.zeros_like(rmse_loss), rmse_loss)))

    mape_loss = torch.abs((preds - targets) / (targets + 1e-5)) * mask
    mape = torch.mean(torch.where(torch.isnan(mape_loss), torch.zeros_like(mape_loss), mape_loss)) * 100

    return mae.item(), rmse.item(), mape.item()

adv_criterion = nn.BCELoss()
LAMBDA_MSE = 100

class GeneratorEarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False
        self.best_state = None

    def __call__(self, val_mse, model):
        if val_mse < self.best_loss - 1e-5:
            self.best_loss = val_mse
            self.counter = 0
            self.best_state = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
            if self.counter >= self.patience: self.early_stop = True





In [ ]:
# ==========================================
# 5. EXPERIMENT EXECUTION (125 LOOPS)
# ==========================================
architectures = list(BLOCKS.keys())
combinations = list(itertools.product(architectures, architectures))

columns = pd.MultiIndex.from_product([scenarios.keys(), ['MAE', 'RMSE', 'MAPE']])
results_df = pd.DataFrame(index=[f"G:{g} + D:{d}" for g, d in combinations], columns=columns)

# Dictionary to store heatmap data for all 5 scenarios
heatmap_data = {scene: np.zeros((5, 5)) for scene in scenarios.keys()}

epochs = 15 # Kept at 15 to ensure the 125 grid search completes in a single Kaggle session
BATCH_SIZE = 128
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

best_model_preds = None
best_model_targets = None
global_best_mae = float('inf')
best_model_name = ""

for exp_name, X_masked in scenarios.items():
    print(f"\n{'='*50}\nSCENARIO: {exp_name.upper()}\n{'='*50}")

    split_idx = int(len(X_masked) * 0.8)
    X_train, Y_train = X_masked[:split_idx].to(device), Y_full[:split_idx].to(device)
    X_test, Y_test = X_masked[split_idx:].to(device), Y_full[split_idx:].to(device)

    for idx, (gen_type, disc_type) in enumerate(combinations):
        row_name = f"G:{gen_type} + D:{disc_type}"
        print(f"Training [{idx+1}/25] -> {row_name}")

        netG = TrafficGenerator(gen_type).to(device)
        netD = TrafficDiscriminator(disc_type).to(device)

        optG = torch.optim.Adam(netG.parameters(), lr=0.001)
        optD = torch.optim.Adam(netD.parameters(), lr=0.001)
        early_stopper = GeneratorEarlyStopping(patience=3) # Faster cutoff for grid search

        for epoch in range(epochs):
            netG.train(); netD.train()

            for i in range(0, len(X_train), BATCH_SIZE):
                batch_X = X_train[i:i+BATCH_SIZE]
                batch_Y = Y_train[i:i+BATCH_SIZE]

                real_labels = torch.ones(batch_X.size(0), 1).to(device)
                fake_labels = torch.zeros(batch_X.size(0), 1).to(device)

                # --- TRAIN DISCRIMINATOR ---
                optD.zero_grad()
                real_seq = torch.cat([batch_X, batch_Y], dim=1)
                pred_real = netD(real_seq)
                lossD_real = adv_criterion(pred_real, real_labels)

                fake_step = netG(batch_X)
                fake_seq = torch.cat([batch_X, fake_step.detach()], dim=1)
                pred_fake = netD(fake_seq)
                lossD_fake = adv_criterion(pred_fake, fake_labels)

                lossD = lossD_real + lossD_fake
                lossD.backward()
                optD.step()

                # --- TRAIN GENERATOR ---
                optG.zero_grad()
                fake_seq_G = torch.cat([batch_X, fake_step], dim=1)
                pred_fake_G = netD(fake_seq_G)
                lossG_adv = adv_criterion(pred_fake_G, real_labels)
                lossG_mse = masked_mse_loss(fake_step, batch_Y)

                lossG = lossG_adv + (LAMBDA_MSE * lossG_mse)
                lossG.backward()
                optG.step()

            # Validation Pass
            netG.eval()
            with torch.no_grad():
                val_fake_step = netG(X_test[:512])
                val_mse = masked_mse_loss(val_fake_step, Y_test[:512])

            early_stopper(val_mse.item(), netG)
            if early_stopper.early_stop:
                break

        # --- FINAL EVALUATION ---
        netG.load_state_dict(early_stopper.best_state)
        netG.eval()
        with torch.no_grad():
            test_preds_scaled = netG(X_test[:1024]) # Test on larger subset for the final report

            test_preds = (test_preds_scaled * (data_max - data_min)) + data_min
            Y_test_real = (Y_test[:1024] * (data_max - data_min)) + data_min

            mae, rmse, mape = calculate_metrics(test_preds, Y_test_real)

        results_df.loc[row_name, (exp_name, 'MAE')] = round(mae, 3)
        results_df.loc[row_name, (exp_name, 'RMSE')] = round(rmse, 3)
        results_df.loc[row_name, (exp_name, 'MAPE')] = round(mape, 3)

        # Save metrics to heatmap matrices
        g_idx = architectures.index(gen_type)
        d_idx = architectures.index(disc_type)
        heatmap_data[exp_name][g_idx, d_idx] = mae

        # Capture the overall best model for the line plot (Evaluated on 0% missing)
        if exp_name == 'missing_0' and mae < global_best_mae:
            global_best_mae = mae
            best_model_preds = test_preds.cpu().numpy()
            best_model_targets = Y_test_real.cpu().numpy()
            best_model_name = row_name

In [ ]:
# ==========================================
# 6. REPORT VISUALIZATION SUITE
# ==========================================
print("\n" + "="*80)
print(" FINAL GAN ARCHITECTURE BENCHMARKING REPORT")
print("="*80)
# Save the table to a CSV for your report
results_df.to_csv('gan_benchmark_results.csv')
print("Metrics saved to 'gan_benchmark_results.csv'.")

# --- Plot 1: The 5-Scenario Heatmap Grid ---
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

for idx, (scene, matrix) in enumerate(heatmap_data.items()):
    sns.heatmap(matrix, annot=True, fmt=".2f", cmap="YlGnBu",
                xticklabels=architectures, yticklabels=architectures, ax=axes[idx], cbar=False)
    axes[idx].set_title(f'Missing Data: {scene.split("_")[1]}% (MAE)', fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Discriminator')
    axes[idx].set_ylabel('Generator')

# Hide the empty 6th subplot
axes[5].axis('off')
plt.suptitle('GAN Architecture Performance Degradation Heatmaps', fontsize=18, fontweight='bold')
plt.tight_layout()
plt.savefig('heatmap_grid.png', dpi=300)
plt.show()

# --- Plot 2: Actual vs Predicted Traffic Wave [Image of Actual vs Predicted Time Series for Traffic Forecasting] ---
if best_model_preds is not None:
    plt.figure(figsize=(16, 5))
    sensor_idx = 10
    time_window = 100

    plt.plot(best_model_targets[:time_window, 0, sensor_idx], label='Ground Truth (Actual Speed)', color='black', alpha=0.8, linewidth=2)
    plt.plot(best_model_preds[:time_window, 0, sensor_idx], label=f'Predicted ({best_model_name})', color='orange', linestyle='dashed', linewidth=2.5)

    plt.title(f'Spatio-Temporal Forecasting Accuracy - Sensor {sensor_idx}', fontsize=16, fontweight='bold')
    plt.xlabel('Time Steps (5-min intervals)', fontsize=12)
    plt.ylabel('Speed (mph)', fontsize=12)
    plt.legend(loc='lower right', fontsize=11)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.savefig('forecasting_wave.png', dpi=300)
    plt.show()